# 🚀 AWS ETL Pipeline: On-Premises SQL Server → S3 → Redshift

**Author:** Vanamala Bhargav | Data Engineer  
**Stack:** AWS Glue · Amazon S3 · Amazon Redshift · PySpark · AWS Step Functions · AWS DMS  
**Use Case:** Enterprise cloud migration pipeline for state government data

---

## 📌 Pipeline Overview

```
On-Prem SQL Server
      │
      ▼  (AWS DMS — CDC Replication)
  S3 Raw Zone  (s3://bucket/raw/)
      │
      ▼  (AWS Glue ETL Job — PySpark)
  S3 Curated Zone  (s3://bucket/curated/)
      │
      ▼  (AWS Glue → COPY Command)
  Amazon Redshift  (Data Warehouse)
      │
      ▼
  Tableau / Power BI
```

## 🎯 What This Notebook Covers
1. Simulate raw data ingestion (on-prem source)
2. Apply Glue-style PySpark transformations
3. Partition & write curated Parquet to S3
4. Load into Redshift via COPY command
5. Data quality validation
6. Performance tuning techniques

## 🔧 Step 1: Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install pyspark boto3 pandas pyarrow faker

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import json
from datetime import datetime, timedelta
import random
from faker import Faker

fake = Faker()
print('✅ Dependencies loaded successfully')

## ⚡ Step 2: Initialize SparkSession (AWS Glue style)

In [ ]:
# In AWS Glue, SparkContext is initialized automatically.
# Locally we configure it to mirror Glue's behavior.

spark = SparkSession.builder \
    .appName('AWS-Glue-ETL-Pipeline') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'✅ Spark version: {spark.version}')
print(f'✅ SparkSession initialized — App: {spark.sparkContext.appName}')

## 📥 Step 3: Simulate On-Premises Source Data (SQL Server)

In [ ]:
# Simulate raw government transactions data from on-prem SQL Server
# In production: AWS DMS replicates this to S3 raw zone

random.seed(42)
Faker.seed(42)

DEPARTMENTS = ['Finance', 'Health', 'Transportation', 'Education', 'Public Safety']
STATES = ['NM', 'TX', 'AZ', 'CO', 'UT']
STATUSES = ['APPROVED', 'PENDING', 'REJECTED', 'PROCESSED']
CATEGORIES = ['Grant', 'Procurement', 'Payroll', 'Operations', 'Capital Expenditure']

def generate_transactions(n=50000):
    records = []
    base_date = datetime(2023, 1, 1)
    for i in range(n):
        txn_date = base_date + timedelta(days=random.randint(0, 730))
        records.append({
            'transaction_id':    f'TXN-{100000 + i}',
            'department':        random.choice(DEPARTMENTS),
            'category':          random.choice(CATEGORIES),
            'amount':            round(random.uniform(500, 500000), 2),
            'status':            random.choice(STATUSES),
            'vendor_name':       fake.company(),
            'vendor_state':      random.choice(STATES),
            'created_date':      txn_date.strftime('%Y-%m-%d'),
            'fiscal_year':       2023 if txn_date.month < 7 else 2024,
            'fiscal_quarter':    f'Q{((txn_date.month - 1) // 3) + 1}',
            'created_by':        fake.name(),
            'last_modified':     (txn_date + timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d %H:%M:%S'),
            'is_deleted':        random.choice([0, 0, 0, 0, 1]),   # 20% soft-deleted
            'source_system':     'SQL_SERVER_PROD',
        })
    return records

raw_data = generate_transactions(50000)
raw_df = spark.createDataFrame(raw_data)
print(f'✅ Generated {raw_df.count():,} raw records from source system')
print(f'📊 Schema:')
raw_df.printSchema()

## 🔍 Step 4: Raw Zone — Data Profiling & Quality Check

In [ ]:
print('=== RAW DATA PROFILE ===')
print(f'Total Records:     {raw_df.count():,}')
print(f'Partitions:        {raw_df.rdd.getNumPartitions()}')
print(f'Columns:           {len(raw_df.columns)}')
print()

# Check for nulls
print('=== NULL COUNTS ===')
null_counts = raw_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in raw_df.columns
])
null_counts.show(vertical=True)

# Distribution by department
print('=== RECORDS BY DEPARTMENT ===')
raw_df.groupBy('department').count().orderBy(F.desc('count')).show()

# Status distribution
print('=== RECORDS BY STATUS ===')
raw_df.groupBy('status').count().orderBy(F.desc('count')).show()

## 🔄 Step 5: Glue ETL Transformations (Raw → Curated)

In [ ]:
from pyspark.sql.functions import (
    col, when, trim, upper, lower, to_date, to_timestamp,
    year, month, dayofmonth, quarter, round as spark_round,
    regexp_replace, lit, current_timestamp, sha2, concat_ws, md5
)

def transform_raw_to_curated(df):
    """
    AWS Glue ETL transformation logic.
    Mirrors production Glue job: raw-to-curated-transactions
    """
    return (
        df
        # ── Filter soft-deleted records
        .filter(col('is_deleted') == 0)

        # ── Standardize string fields
        .withColumn('department',    upper(trim(col('department'))))
        .withColumn('category',      upper(trim(col('category'))))
        .withColumn('status',        upper(trim(col('status'))))
        .withColumn('vendor_name',   trim(col('vendor_name')))
        .withColumn('vendor_state',  upper(trim(col('vendor_state'))))

        # ── Cast date/timestamp types
        .withColumn('created_date',    to_date(col('created_date'), 'yyyy-MM-dd'))
        .withColumn('last_modified',   to_timestamp(col('last_modified'), 'yyyy-MM-dd HH:mm:ss'))

        # ── Derive date parts for partitioning
        .withColumn('txn_year',    year(col('created_date')))
        .withColumn('txn_month',   month(col('created_date')))
        .withColumn('txn_quarter', quarter(col('created_date')))
        .withColumn('txn_day',     dayofmonth(col('created_date')))

        # ── Amount bucketing for analytics
        .withColumn('amount_bucket',
            when(col('amount') < 10000,               'SMALL')
            .when(col('amount').between(10000, 99999), 'MEDIUM')
            .when(col('amount').between(100000, 249999),'LARGE')
            .otherwise('ENTERPRISE')
        )

        # ── Approved flag
        .withColumn('is_approved',
            when(col('status') == 'APPROVED', 1).otherwise(0)
        )

        # ── Surrogate key (SHA-256 hash)
        .withColumn('record_hash',
            sha2(concat_ws('|', col('transaction_id'), col('last_modified')), 256)
        )

        # ── ETL audit columns
        .withColumn('etl_load_timestamp', current_timestamp())
        .withColumn('etl_source', lit('S3_RAW_ZONE'))

        # ── Drop columns not needed downstream
        .drop('is_deleted', 'source_system')
    )

curated_df = transform_raw_to_curated(raw_df)

print(f'✅ Raw records:     {raw_df.count():,}')
print(f'✅ Curated records: {curated_df.count():,} (after filtering deleted)')
print(f'📊 New columns added: amount_bucket, is_approved, record_hash, etl_*')
curated_df.printSchema()

## 📊 Step 6: Data Quality Validation (Pre-Load Checks)

In [ ]:
def run_dq_checks(df, stage='CURATED'):
    """
    Data Quality checks mirroring AWS Glue Data Quality rules.
    In production these feed CloudWatch metrics and SNS alerts.
    """
    checks = {}

    # Rule 1: No nulls in primary key
    null_pk = df.filter(col('transaction_id').isNull()).count()
    checks['no_null_transaction_id'] = ('PASS' if null_pk == 0 else 'FAIL', null_pk)

    # Rule 2: Amount must be positive
    neg_amounts = df.filter(col('amount') <= 0).count()
    checks['positive_amount'] = ('PASS' if neg_amounts == 0 else 'FAIL', neg_amounts)

    # Rule 3: Status in allowed values
    valid_statuses = ['APPROVED', 'PENDING', 'REJECTED', 'PROCESSED']
    invalid_status = df.filter(~col('status').isin(valid_statuses)).count()
    checks['valid_status_values'] = ('PASS' if invalid_status == 0 else 'FAIL', invalid_status)

    # Rule 4: No duplicate transaction_ids
    total = df.count()
    distinct = df.select('transaction_id').distinct().count()
    checks['no_duplicate_txn_ids'] = ('PASS' if total == distinct else 'FAIL', total - distinct)

    # Rule 5: Date range sanity (2020–2025)
    bad_dates = df.filter(
        (year(col('created_date')) < 2020) | (year(col('created_date')) > 2025)
    ).count()
    checks['valid_date_range'] = ('PASS' if bad_dates == 0 else 'FAIL', bad_dates)

    print(f'\n=== DATA QUALITY REPORT — {stage} ZONE ===')
    all_pass = True
    for rule, (result, count) in checks.items():
        icon = '✅' if result == 'PASS' else '❌'
        print(f'  {icon} {rule:<35} [{result}]  violations: {count}')
        if result == 'FAIL':
            all_pass = False

    print(f'\n  Overall: {"✅ ALL CHECKS PASSED" if all_pass else "❌ SOME CHECKS FAILED"}')
    return all_pass

dq_passed = run_dq_checks(curated_df)

## ⚡ Step 7: PySpark Performance Tuning

In [ ]:
# ── Technique 1: Optimal Partitioning
# Rule of thumb: 128MB per partition. Repartition by natural key for downstream joins.
optimized_df = curated_df.repartition(8, 'department', 'txn_year')
print(f'Partitions after repartition: {optimized_df.rdd.getNumPartitions()}')

# ── Technique 2: Cache hot dataset (used in multiple downstream steps)
optimized_df.cache()
optimized_df.count()  # Trigger cache
print('✅ Dataset cached in memory')

# ── Technique 3: Broadcast join for small dimension tables
dept_budget = spark.createDataFrame([
    ('FINANCE',         50000000),
    ('HEALTH',          75000000),
    ('TRANSPORTATION',  40000000),
    ('EDUCATION',       60000000),
    ('PUBLIC SAFETY',   35000000),
], ['department', 'annual_budget'])

# Broadcast the small dimension table (< 10MB) to avoid shuffle
enriched_df = optimized_df.join(
    F.broadcast(dept_budget),   # <-- Broadcast hint
    on='department',
    how='left'
).withColumn(
    'pct_of_budget',
    spark_round((col('amount') / col('annual_budget')) * 100, 4)
)

print('✅ Broadcast join completed — no shuffle required')
enriched_df.select('transaction_id','department','amount','annual_budget','pct_of_budget').show(5)

## 💾 Step 8: Write Curated Data to S3 (Partitioned Parquet)

In [ ]:
# In AWS Glue this writes to: s3://your-bucket/curated/transactions/
# Locally we write to ./data/curated/ to demonstrate the pattern

OUTPUT_PATH = './data/curated/transactions'

(
    enriched_df
    .write
    .mode('overwrite')
    .partitionBy('txn_year', 'txn_month')    # S3 partition layout
    .option('compression', 'snappy')          # Snappy for Redshift COPY
    .parquet(OUTPUT_PATH)
)

# Verify output
verify_df = spark.read.parquet(OUTPUT_PATH)
print(f'✅ Written to: {OUTPUT_PATH}')
print(f'📊 Rows written: {verify_df.count():,}')
print(f'📁 Partitions:  txn_year / txn_month')
verify_df.show(3)

## 🏗️ Step 9: Redshift Load — COPY Command Simulation

In [ ]:
# In production, AWS Glue uses the Redshift connector or COPY command via psycopg2
# This cell shows the exact SQL and logic used

REDSHIFT_COPY_SQL = """
-- Step 1: Stage into temp table
CREATE TEMP TABLE stg_transactions (LIKE fact_transactions);

COPY stg_transactions
FROM 's3://nm-state-datalake/curated/transactions/'
IAM_ROLE 'arn:aws:iam::123456789:role/RedshiftS3Role'
FORMAT AS PARQUET
SERIALIZETOJSON;

-- Step 2: Upsert into fact table (DELETE + INSERT)
DELETE FROM fact_transactions
WHERE transaction_id IN (SELECT transaction_id FROM stg_transactions);

INSERT INTO fact_transactions
SELECT * FROM stg_transactions;

-- Step 3: Vacuum & Analyze
VACUUM fact_transactions;
ANALYZE fact_transactions;
"""

# Redshift DDL
REDSHIFT_DDL = """
CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id      VARCHAR(20)     NOT NULL,
    department          VARCHAR(50)     NOT NULL,
    category            VARCHAR(50),
    amount              DECIMAL(15,2),
    status              VARCHAR(20),
    vendor_name         VARCHAR(200),
    vendor_state        VARCHAR(5),
    created_date        DATE,
    fiscal_year         INTEGER,
    fiscal_quarter      VARCHAR(3),
    amount_bucket       VARCHAR(15),
    is_approved         SMALLINT,
    pct_of_budget       DECIMAL(10,4),
    record_hash         VARCHAR(64),
    etl_load_timestamp  TIMESTAMP,
    etl_source          VARCHAR(50)
)
DISTSTYLE KEY
DISTKEY(department)
SORTKEY(created_date, department);
"""

print('📋 Redshift DDL:')
print(REDSHIFT_DDL)
print('📋 Redshift COPY Command:')
print(REDSHIFT_COPY_SQL)

## 📊 Step 10: Analytics Queries on Curated Data

In [ ]:
# Register temp view (mirrors Redshift analytical queries)
enriched_df.createOrReplaceTempView('fact_transactions')

# Query 1: Total spend by department and fiscal year
print('=== SPEND BY DEPARTMENT & FISCAL YEAR ===')
spark.sql("""
    SELECT
        department,
        fiscal_year,
        COUNT(*)                                    AS txn_count,
        ROUND(SUM(amount), 2)                       AS total_spend,
        ROUND(AVG(amount), 2)                       AS avg_txn_amount,
        SUM(is_approved)                            AS approved_count,
        ROUND(SUM(is_approved) * 100.0 / COUNT(*), 1) AS approval_rate_pct
    FROM fact_transactions
    GROUP BY department, fiscal_year
    ORDER BY fiscal_year, total_spend DESC
""").show(20)

# Query 2: Amount bucket distribution
print('=== TRANSACTION SIZE DISTRIBUTION ===')
spark.sql("""
    SELECT
        amount_bucket,
        COUNT(*)                       AS txn_count,
        ROUND(SUM(amount), 0)          AS total_value,
        ROUND(AVG(amount), 0)          AS avg_value
    FROM fact_transactions
    GROUP BY amount_bucket
    ORDER BY total_value DESC
""").show()

# Query 3: Monthly trend
print('=== MONTHLY SPEND TREND ===')
spark.sql("""
    SELECT
        txn_year, txn_month,
        ROUND(SUM(amount), 0) AS monthly_spend,
        COUNT(*) AS txn_count
    FROM fact_transactions
    GROUP BY txn_year, txn_month
    ORDER BY txn_year, txn_month
""").show(24)

## ✅ Step 11: Pipeline Summary

| Step | Component | Status |
|---|---|---|
| 1 | Source simulation (SQL Server) | ✅ |
| 2 | SparkSession (AWS Glue style) | ✅ |
| 3 | PySpark transformations | ✅ |
| 4 | Data quality checks | ✅ |
| 5 | Performance tuning (broadcast, cache, partition) | ✅ |
| 6 | Write partitioned Parquet to S3 | ✅ |
| 7 | Redshift COPY + DDL | ✅ |
| 8 | Analytical SQL queries | ✅ |

### 🔗 Production AWS Components
- **AWS DMS** → CDC from SQL Server to S3 Raw
- **AWS Glue Crawler** → Auto-catalog S3 partitions in Glue Data Catalog
- **AWS Glue Job** → This notebook's PySpark logic
- **AWS Step Functions** → Orchestrates Glue → Lambda → Redshift COPY
- **CloudWatch** → Pipeline monitoring and alerting
- **Terraform** → Infrastructure as Code deployment